## 문자 단위 RNN(Char RNN)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

### 훈련 데이터 전처리

In [ ]:
sentence = ("if you want to build a ship, don't drum up people together to "
            "collect wood and don't assign them tasks and work, but rather "
            "teach them to long for the endless immensity of the sea.")

문자 집합을 생성하고, 각 문자에 고유한 정수를 부여.

In [ ]:
char_set = list(set(sentence)) # 중복을 제거한 문자 집합 생성
char_dic = {c: i for i, c in enumerate(char_set)} # 각 문자에 정수 인코딩

In [ ]:
print(char_dic) # 공백도 여기서는 하나의 원소

{'o': 0, 'n': 1, 'p': 2, 'h': 3, 'r': 4, 'm': 5, 'g': 6, 'w': 7, '.': 8, "'": 9, 'f': 10, 'i': 11, 'e': 12, 'l': 13, 't': 14, 'b': 15, ',': 16, ' ': 17, 's': 18, 'y': 19, 'a': 20, 'u': 21, 'k': 22, 'd': 23, 'c': 24}


In [ ]:
dic_size = len(char_dic)
print('문자 집합의 크기 : {}'.format(dic_size))

문자 집합의 크기 : 25


하이퍼파라미터 설정

In [ ]:
sequence_length = 10  # 임의 숫자 지정, RNN의 입력에 사용할 문자 수

sequence_length에 맞추어 학습 데이터를 입력과 출력으로 나눔

In [ ]:
# 데이터 구성
x_data = []
y_data = []

for i in range(0, len(sentence) - sequence_length):
    x_str = sentence[i:i + sequence_length]
    y_str = sentence[i + 1: i + sequence_length + 1]
    print(i, x_str, '->', y_str)

    x_data.append([char_dic[c] for c in x_str])  # x str to index
    y_data.append([char_dic[c] for c in y_str])  # y str to index

0 if you wan -> f you want
1 f you want ->  you want 
2  you want  -> you want t
3 you want t -> ou want to
4 ou want to -> u want to 
5 u want to  ->  want to b
6  want to b -> want to bu
7 want to bu -> ant to bui
8 ant to bui -> nt to buil
9 nt to buil -> t to build
10 t to build ->  to build 
11  to build  -> to build a
12 to build a -> o build a 
13 o build a  ->  build a s
14  build a s -> build a sh
15 build a sh -> uild a shi
16 uild a shi -> ild a ship
17 ild a ship -> ld a ship,
18 ld a ship, -> d a ship, 
19 d a ship,  ->  a ship, d
20  a ship, d -> a ship, do
21 a ship, do ->  ship, don
22  ship, don -> ship, don'
23 ship, don' -> hip, don't
24 hip, don't -> ip, don't 
25 ip, don't  -> p, don't d
26 p, don't d -> , don't dr
27 , don't dr ->  don't dru
28  don't dru -> don't drum
29 don't drum -> on't drum 
30 on't drum  -> n't drum u
31 n't drum u -> 't drum up
32 't drum up -> t drum up 
33 t drum up  ->  drum up p
34  drum up p -> drum up pe
35 drum up pe -> rum up peo
36

정수 인코딩 확인

In [ ]:
print(x_data[0]) # if you wan
print(y_data[0]) # f you want

[11, 10, 17, 19, 0, 21, 17, 7, 20, 1]
[10, 17, 19, 0, 21, 17, 7, 20, 1, 14]


입력은 원-핫 벡터로 변경

In [ ]:
x_one_hot = [np.eye(dic_size)[x] for x in x_data] # x 데이터는 원-핫 인코딩
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

In [ ]:
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))

훈련 데이터의 크기 : torch.Size([170, 10, 25])
레이블의 크기 : torch.Size([170, 10])


In [ ]:
print(X[0]) # if you wan의 원-핫 벡터

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,

In [ ]:
print(Y[0]) # f you want

tensor([10, 17, 19,  0, 21, 17,  7, 20,  1, 14])


### 모델 구현

In [ ]:
class Net(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, dic_size, layers): # 현재 hidden_size는 dic_size와 같음.
        super(Net, self).__init__()

        # RNN Layer
        self.rnn = torch.nn.RNN(input_dim, hidden_dim, num_layers=layers, batch_first=True)
        # FC Layer
        self.fc = torch.nn.Linear(hidden_dim, dic_size, bias=True)

    def forward(self, x):
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x

In [ ]:
hidden_size = dic_size
learning_rate = 0.1
num_layers = 2

net = Net(dic_size, hidden_size, dic_size, num_layers)

In [ ]:
outputs = net(X)
print(outputs.shape) # network의 출력은? 3차원 텐서 [배치 차원, 시점, 출력 크기]

torch.Size([170, 10, 25])


In [ ]:
print(outputs.view(-1, dic_size).shape) # 출력을 2차원 텐서로 변환.

torch.Size([1700, 25])


In [ ]:
print(Y.shape)  # label의 크기
print(Y.view(-1).shape) # label을 펼치면

torch.Size([170, 10])
torch.Size([1700])


In [ ]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [ ]:
for i in range(100):
    # Model Forward Pass
    # (170, 10, 25) 크기를 가진 텐서를 매 에포크마다 모델의 입력으로 사용
    outputs = net(X)

    # Loss: (outputs.view(-1, dic_size)와 Y.view(-1) 사용 필요
    loss = criterion(outputs.view(-1, dic_size), Y.view(-1))

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # results의 텐서 크기는 (170, 10)
    results = outputs.argmax(dim=2)
    predict_str = "" # 예측 시작 character
    for j, result in enumerate(results):
        if j == 0: # 처음에는 예측 결과를 전부 가져오지만
            predict_str += ''.join([char_set[t] for t in result])
        else: # 그 다음에는 마지막 글자만 반복 추가
            predict_str += char_set[result[-1]]

    print(predict_str)

euuu'uu'u,'u'uuu''uue''uueuueu',''u'ee'u''euu,u''e'uu''uuee'uu''u'ueeu'uuu''au'''u''u'euuueu'uuee''eu'u'au''uu'''uu'ueeuuuee''eeuu'uuee''u'','uuueu'uue'''''euu''eu,''uue'uu'uueuu,
 r                r       r dr     r   d      r r       r d r     r      r    dr                d   r     d     r   r   rr d r        d    r   r    r  r   r m  r r     d     r  r 
tt tt tt  t t   t o t t t e t t t t  t t t    t t t t     t   t t     t t t t t   t t t tt t   tt t t t   t t t t t t t e  t t t t t  t ttt tt    t t t   tt t  t t t t t t t t t  
di.do .do ........, ... .g..... ,.... .c........... .rg..p...., ... .... ..., ... .c... ..... ... ..g ... . . . ... ...d..... ..g... .. ..g .ic.. ...... ..g .. ....... .  .... ...
ts ds d ,s dnrdr, , , , , , ,sr , a  d,ad a,r dsnbs, rna  ,rnb, e. d a,ndran, , , , a. da,rd,or ds d rs , ,sr , , , ,snp , brr , a  , as dsn radb d as ,sr, d as, , as dsndp d air 
en ts bon  bonp o t t t t t t n t to  ton i n tonpis t to bonpt  in  s  pbs t t n t to  tn tt tegt t